# E-commerce Marketplace Financial Analysis

Analyzing sales data and payment status from marketplace

In [1]:
import pandas as pd
import numpy as np

# Load the CSV files with semicolon delimiter and handle German number format
df_gmu = pd.read_csv(
    'report_booking_gmu_de_9de5e3ee9d9d950a2afba4b21aa6029fc8616843ea00a471bc76f03ed5d8c2e8.csv',
    sep=';',
    decimal=',',
    thousands='.',
    encoding='utf-8'
)

df_10000 = pd.read_csv(
    'report_booking10000_de_df2ac594159928a30cbdd344e7772459a8d196693f3c96288873dc316ef2bab5.csv',
    sep=';',
    decimal=',',
    thousands='.',
    encoding='utf-8'
)

# Convert dates to datetime
df_gmu['booking_date'] = pd.to_datetime(df_gmu['booking_date'])
df_10000['Datum'] = pd.to_datetime(df_10000['Datum'])

# Add month column
df_gmu['month'] = df_gmu['booking_date'].dt.to_period('M')
df_10000['month'] = df_10000['Datum'].dt.to_period('M')

print("✓ Datasets loaded successfully")
print(f"GMU: {df_gmu.shape[0]} rows, {df_gmu.shape[1]} columns")
print(f"  Date range: {df_gmu['booking_date'].min()} to {df_gmu['booking_date'].max()}")
print(f"df_10000: {df_10000.shape[0]} rows, {df_10000.shape[1]} columns")
print(f"  Date range: {df_10000['Datum'].min()} to {df_10000['Datum'].max()}")

✓ Datasets loaded successfully
GMU: 44 rows, 54 columns
  Date range: 2025-08-05 08:53:44 to 2025-10-27 08:37:17
df_10000: 101 rows, 9 columns
  Date range: 2025-08-05 00:00:00 to 2025-10-25 00:00:00


In [2]:
# Understanding the data structure
print("="*70)
print("UNDERSTANDING THE DATA STRUCTURE")
print("="*70)

print("\n1. GMU Dataset - Transaction Types:")
print(df_gmu['booking_text'].value_counts().head(10))

print("\n2. df_10000 Dataset - Transaction Types:")
print(df_10000['Buchungstext'].value_counts().head(10))

print("\n3. Understanding Payment Flow:")
print("  - Wareneingang (Sales Income): Order placed, positive amount")
print("  - Netto Provision (Commission): Commission charged, negative amount")
print("  - Freigabe Verkaufserlös (Sales Released): PAYMENT RECEIVED, negative amount")
print("  - An order is PAID when 'Freigabe' transaction exists in df_10000")

UNDERSTANDING THE DATA STRUCTURE

1. GMU Dataset - Transaction Types:
booking_text
Payout                                                                                                                10
Fees for cancelled orders Juli 25                                                                                      2
Bezahlung Grundgebühr                                                                                                  2
Freigabe Verkaufserlös zu Bestell-Nr. MG6CC7Q/314567987448311 Injusa 6v 606 Ce E230 S6/0.6 C1 Batterie Schwarz ...     1
Freigabe Verkaufserlös zu Bestell-Nr. MX6XA6Q/314567986969924 Safta Mufasa Big Triple Federmäppchen Grün Mann G...     1
Freigabe Verkaufserlös zu Bestell-Nr. M5QHW6Q/314567987029312 Harry Potter Regenschirm ? 97 cm Leichter Kuppel-...     1
Freigabe Verkaufserlös zu Bestell-Nr. MZPEE6Q/314567986723309 Federmäppchen mit Zubehör Peppa Pig Pretty flower...     1
Freigabe Verkaufserlös zu Bestell-Nr. MY7ZB7Q/314567987437097 Kinder-K

In [3]:
# Identify which orders have been PAID (have Freigabe transaction)
print("="*70)
print("IDENTIFYING PAID VS UNPAID ORDERS")
print("="*70)

# Get unique order numbers from both datasets
gmu_orders = set(df_gmu['order_number'].dropna().unique())
df10000_all_orders = set(df_10000['Bestellnummer'].dropna().unique())

# Orders are PAID only if they have a 'Freigabe' (Sales Released) transaction
freigabe_orders = set(df_10000[df_10000['Buchungstext'].str.contains('Freigabe', na=False)]['Bestellnummer'].dropna().unique())

# Orders with Wareneingang (created) in df_10000
wareneingang_orders = set(df_10000[df_10000['Buchungstext'].str.contains('Wareneingang', na=False)]['Bestellnummer'].dropna().unique())

print(f"\nUnique orders in GMU: {len(gmu_orders)}")
print(f"Unique orders in df_10000: {len(df10000_all_orders)}")
print(f"Orders with Wareneingang (created): {len(wareneingang_orders)}")
print(f"Orders with Freigabe (PAID): {len(freigabe_orders)}")

# UNPAID = Orders with Wareneingang but NO Freigabe
unpaid_order_numbers_df10000 = wareneingang_orders - freigabe_orders
unpaid_order_numbers_gmu = gmu_orders - freigabe_orders

print(f"\nUnpaid orders (in df_10000): {len(unpaid_order_numbers_df10000)}")
print(f"Unpaid orders (in GMU): {len(unpaid_order_numbers_gmu)}")
print(f"Paid orders: {len(freigabe_orders)}")

if len(unpaid_order_numbers_df10000) > 0:
    print(f"\nSample unpaid order numbers: {list(unpaid_order_numbers_df10000)[:5]}")

IDENTIFYING PAID VS UNPAID ORDERS

Unique orders in GMU: 29
Unique orders in df_10000: 42
Orders with Wareneingang (created): 33
Orders with Freigabe (PAID): 29

Unpaid orders (in df_10000): 13
Unpaid orders (in GMU): 0
Paid orders: 29

Sample unpaid order numbers: ['MHFBPSQ', 'M5KWE6Q', 'M77T4GQ', 'MDWCPGQ', 'M6R54GQ']


In [4]:
# Create comprehensive order list from BOTH sources
print("="*70)
print("BUILDING COMPLETE ORDER LIST")
print("="*70)

# Start with GMU sales
gmu_sales = df_gmu[df_gmu['order_number'].notna()].copy()
gmu_sales['is_paid'] = gmu_sales['order_number'].isin(freigabe_orders)
gmu_sales['source'] = 'GMU'

print(f"\nOrders from GMU: {len(gmu_sales)}")
print(f"  Paid: {gmu_sales['is_paid'].sum()}")
print(f"  Unpaid: {(~gmu_sales['is_paid']).sum()}")

# Find orders in df_10000 that are NOT in GMU (recent orders)
orders_only_in_df10000 = wareneingang_orders - gmu_orders
print(f"\nOrders ONLY in df_10000 (not yet in GMU): {len(orders_only_in_df10000)}")

if len(orders_only_in_df10000) > 0:
    print(f"  These are recent orders: {list(orders_only_in_df10000)[:5]}")
    
    # Create entries for these orders from df_10000 data
    wareneingang_df = df_10000[df_10000['Buchungstext'].str.contains('Wareneingang', na=False)].copy()
    commission_df = df_10000[df_10000['Buchungstext'].str.contains('Provision', na=False)].copy()
    
    recent_orders_list = []
    for order_num in orders_only_in_df10000:
        waren_row = wareneingang_df[wareneingang_df['Bestellnummer'] == order_num]
        comm_row = commission_df[commission_df['Bestellnummer'] == order_num]
        
        if len(waren_row) > 0:
            row = waren_row.iloc[0]
            commission = abs(comm_row.iloc[0]['Betrag']) if len(comm_row) > 0 else 0
            gross = row['Betrag']
            
            recent_orders_list.append({
                'order_number': order_num,
                'booking_date': row['Datum'],
                'order_date': row['Datum'],
                'booking_text': row['Buchungstext'],
                'price_gross': gross,
                'sum_price_gross': gross,
                'fee_gross': commission,
                'payout': gross - commission,
                'is_paid': order_num in freigabe_orders,
                'source': 'df_10000_only',
                'title_item': 'See df_10000 for details',
                'shipping_charges_gross': 0,
                'fee_%': (commission / gross * 100) if gross > 0 else 0,
                'buyer.email': '',
                'shipping.first_name': '',
                'shipping.last_name': '',
                'shipping.city': ''
            })
    
    recent_orders_df = pd.DataFrame(recent_orders_list)
    print(f"  Created {len(recent_orders_df)} entries from df_10000")
    
    # Combine with GMU sales
    all_orders = pd.concat([gmu_sales, recent_orders_df], ignore_index=True)
else:
    all_orders = gmu_sales

# Final counts
paid_orders = all_orders[all_orders['is_paid']].copy()
unpaid_orders = all_orders[~all_orders['is_paid']].copy()

print(f"\nFINAL COUNTS:")
print(f"  Total orders: {len(all_orders)}")
print(f"  Paid: {len(paid_orders)}")
print(f"  Unpaid: {len(unpaid_orders)}")

BUILDING COMPLETE ORDER LIST

Orders from GMU: 29
  Paid: 29
  Unpaid: 0

Orders ONLY in df_10000 (not yet in GMU): 13
  These are recent orders: ['MHFBPSQ', 'M5KWE6Q', 'M77T4GQ', 'MDWCPGQ', 'M6R54GQ']
  Created 13 entries from df_10000

FINAL COUNTS:
  Total orders: 42
  Paid: 29
  Unpaid: 13


In [5]:
# Comprehensive financial summary
print("="*70)
print("FINANCIAL SUMMARY")
print("="*70)

summary = {
    'Total Sales': {
        'Gross Sales': all_orders['price_gross'].sum(),
        'Shipping': all_orders['shipping_charges_gross'].sum(),
        'Total Gross': all_orders['sum_price_gross'].sum(),
        'Commission (Fees)': all_orders['fee_gross'].sum(),
        'Net Payout Expected': all_orders['payout'].sum(),
        'Order Count': len(all_orders)
    },
    'PAID Orders': {
        'Gross Sales': paid_orders['price_gross'].sum(),
        'Shipping': paid_orders['shipping_charges_gross'].sum(),
        'Total Gross': paid_orders['sum_price_gross'].sum(),
        'Commission (Fees)': paid_orders['fee_gross'].sum(),
        'Net Payout Expected': paid_orders['payout'].sum(),
        'Order Count': len(paid_orders)
    },
    'UNPAID Orders': {
        'Gross Sales': unpaid_orders['price_gross'].sum(),
        'Shipping': unpaid_orders['shipping_charges_gross'].sum(),
        'Total Gross': unpaid_orders['sum_price_gross'].sum(),
        'Commission (Fees)': unpaid_orders['fee_gross'].sum(),
        'Net Payout Expected': unpaid_orders['payout'].sum(),
        'Order Count': len(unpaid_orders)
    }
}

summary_df = pd.DataFrame(summary).T
print("\n", summary_df.round(2))

# Actual amount received = sum of all Freigabe transactions (negative values)
actual_received_freigabe = df_10000[df_10000['Buchungstext'].str.contains('Freigabe', na=False)]['Betrag'].sum()
avg_commission_rate = (all_orders['fee_gross'].sum() / all_orders['sum_price_gross'].sum() * 100)

print(f"\n\nActual Payout RECEIVED (Freigabe): {abs(actual_received_freigabe):.2f} EUR")
print(f"Expected Payout (Paid): {paid_orders['payout'].sum():.2f} EUR")
print(f"Expected Payout (Unpaid): {unpaid_orders['payout'].sum():.2f} EUR")
print(f"Average Commission: {avg_commission_rate:.2f}%")

FINANCIAL SUMMARY

                Gross Sales  Shipping  Total Gross  Commission (Fees)  \
Total Sales        1380.58     155.0      1535.58             210.15   
PAID Orders         791.03     155.0       946.03             128.02   
UNPAID Orders       589.55       0.0       589.55              82.13   

               Net Payout Expected  Order Count  
Total Sales                1325.43         42.0  
PAID Orders                 818.01         29.0  
UNPAID Orders               507.42         13.0  


Actual Payout RECEIVED (Freigabe): 818.01 EUR
Expected Payout (Paid): 818.01 EUR
Expected Payout (Unpaid): 507.42 EUR
Average Commission: 13.69%


In [6]:
# Categorize transactions
print("="*70)
print("CATEGORIZING TRANSACTIONS")
print("="*70)

# GMU categories
def categorize_gmu_transaction(row):
    text = str(row['booking_text']).lower()
    if pd.isna(row['order_number']):
        if 'fee' in text or 'storno fee' in text:
            return 'Fees - Cancelled Orders'
        elif 'payout' in text:
            return 'Payout Transfer'
        elif 'grundgebühr' in text or 'bezahlung grundgebühr' in text:
            return 'Grundgebühr (Base Fee)'
        else:
            return 'Other Non-Sales'
    else:
        if 'freigabe' in text:
            return 'Sales - Released'
        else:
            return 'Sales - Other'

df_gmu['transaction_category'] = df_gmu.apply(categorize_gmu_transaction, axis=1)

# df_10000 categories
def categorize_payment_transaction(row):
    text = str(row['Buchungstext']).lower()
    if 'wareneingang' in text:
        return 'Sales Income'
    elif 'provision' in text or 'netto provision' in text:
        return 'Commission/Fees'
    elif 'freigabe' in text:
        return 'Sales Released (PAID)'
    elif 'payout' in text:
        return 'Payout Transfer'
    elif 'grundgebühr' in text or 'bezahlung grundgebühr' in text:
        return 'Grundgebühr (Base Fee)'
    elif 'storno' in text:
        return 'Cancellation/Refund'
    else:
        return 'Other'

df_10000['transaction_category'] = df_10000.apply(categorize_payment_transaction, axis=1)

# Monthly breakdowns
gmu_category_monthly = df_gmu.pivot_table(
    values='payout',
    index='month',
    columns='transaction_category',
    aggfunc='sum',
    fill_value=0
).round(2)

# FIXED: Flip signs for payment categories to make them more readable
df10000_category_monthly = df_10000.pivot_table(
    values='Betrag',
    index='month',
    columns='transaction_category',
    aggfunc='sum',
    fill_value=0
).round(2)

# Flip sign for Sales Released (make it positive for readability)
if 'Sales Released (PAID)' in df10000_category_monthly.columns:
    df10000_category_monthly['Sales Released (PAID)'] = df10000_category_monthly['Sales Released (PAID)'].abs()

# Flip sign for Commission/Fees (make it positive for readability)
if 'Commission/Fees' in df10000_category_monthly.columns:
    df10000_category_monthly['Commission/Fees'] = df10000_category_monthly['Commission/Fees'].abs()

print("\n✓ Categories created (with corrected signs)")

CATEGORIZING TRANSACTIONS

✓ Categories created (with corrected signs)


In [7]:
# Identify shipped orders and CANCELLED/RETURNED (including fees)
print("="*70)
print("SHIPPED & CANCELLED ORDERS ANALYSIS")
print("="*70)

# Shipped orders from GMU sales only (not the df_10000 additions)
shipped_orders = gmu_sales[
    (gmu_sales['payout'] > 0) & 
    (gmu_sales['booking_text'].str.contains('Freigabe', na=False))
].copy()
shipped_orders['payment_status'] = shipped_orders['is_paid'].map({True: 'PAID', False: 'UNPAID'})

# CANCELLED/RETURNED: Include both orders AND fees for cancelled orders
cancelled_returned_orders = gmu_sales[
    (gmu_sales['payout'] <= 0) | 
    (gmu_sales['booking_text'].str.contains('Storno|Rückgabe|Cancel|Return', case=False, na=False))
].copy()

# Add cancelled fees (non-order transactions)
cancelled_fees = df_gmu[
    (df_gmu['order_number'].isna()) & 
    (df_gmu['booking_text'].str.contains('Fees for cancelled|Storno', case=False, na=False))
].copy()

print(f"\nShipped orders: {len(shipped_orders)}")
print(f"Cancelled/Returned orders: {len(cancelled_returned_orders)}")
print(f"Cancelled fees (non-order): {len(cancelled_fees)}")
print(f"\nTotal cancelled/fees to report: {len(cancelled_returned_orders) + len(cancelled_fees)}")

SHIPPED & CANCELLED ORDERS ANALYSIS

Shipped orders: 29
Cancelled/Returned orders: 0
Cancelled fees (non-order): 3

Total cancelled/fees to report: 3


In [8]:
# Payment timing analysis
print("="*70)
print("PAYMENT TIMING")
print("="*70)

paid_details = all_orders[all_orders['is_paid']].copy()

# Get dates from df_10000
freigabe_dates = df_10000[df_10000['Buchungstext'].str.contains('Freigabe', na=False)].copy()
freigabe_dates = freigabe_dates.sort_values('Datum').groupby('Bestellnummer')['Datum'].last()

wareneingang_dates = df_10000[df_10000['Buchungstext'].str.contains('Wareneingang', na=False)].copy()
wareneingang_dates = wareneingang_dates.sort_values('Datum').groupby('Bestellnummer')['Datum'].first()

paid_details = paid_details.set_index('order_number')
paid_details['order_created_date'] = paid_details.index.map(wareneingang_dates)
paid_details['payment_received_date'] = paid_details.index.map(freigabe_dates)
paid_details = paid_details.reset_index()

paid_details['payment_delay_days'] = (paid_details['payment_received_date'] - paid_details['order_created_date']).dt.days

print("\nPayment Delay (days from order to payment):")
print(paid_details['payment_delay_days'].describe())

PAYMENT TIMING

Payment Delay (days from order to payment):
count    20.000000
mean     25.300000
std       8.297749
min      19.000000
25%      20.000000
50%      21.500000
75%      24.750000
max      47.000000
Name: payment_delay_days, dtype: float64


In [9]:
# Flag orders with missing/zero amounts
print("="*70)
print("CHECKING FOR MISSING AMOUNTS")
print("="*70)

# Only check GMU orders (not df_10000 additions)
gmu_sales_check = gmu_sales.copy()
gmu_sales_check['has_missing_price'] = (gmu_sales_check['price_gross'].isna()) | (gmu_sales_check['price_gross'] == 0)
gmu_sales_check['has_missing_payout'] = (gmu_sales_check['payout'].isna()) | (gmu_sales_check['payout'] == 0)
gmu_sales_check['has_missing_sum'] = (gmu_sales_check['sum_price_gross'].isna()) | (gmu_sales_check['sum_price_gross'] == 0)

problematic_orders = gmu_sales_check[
    gmu_sales_check['has_missing_price'] | 
    gmu_sales_check['has_missing_payout'] | 
    gmu_sales_check['has_missing_sum']
]

print(f"\nOrders with missing/zero amounts: {len(problematic_orders)}")
if len(problematic_orders) == 0:
    print("✓ All amounts are present")

CHECKING FOR MISSING AMOUNTS

Orders with missing/zero amounts: 0
✓ All amounts are present


In [10]:
# Create All Orders Overview
print("="*70)
print("CREATING ALL ORDERS OVERVIEW")
print("="*70)

all_orders_overview = all_orders[[
    'booking_date', 'order_date', 'order_number', 'title_item',
    'price_gross', 'shipping_charges_gross', 'sum_price_gross',
    'fee_gross', 'payout', 'is_paid', 'source'
]].copy()

# Add payment status label
all_orders_overview['payment_status'] = all_orders_overview['is_paid'].map({True: 'PAID', False: 'UNPAID'})

# Add payment received date for paid orders
all_orders_overview = all_orders_overview.set_index('order_number')
all_orders_overview['payment_received_date'] = all_orders_overview.index.map(freigabe_dates)
all_orders_overview = all_orders_overview.reset_index()

# Add days since order
all_orders_overview['days_since_order'] = (pd.Timestamp.now() - all_orders_overview['booking_date']).dt.days

# Sort by payment status (unpaid first) then by date
all_orders_overview = all_orders_overview.sort_values(['is_paid', 'booking_date'], ascending=[True, False])

print(f"\nTotal orders: {len(all_orders_overview)}")
print(f"PAID: {all_orders_overview['is_paid'].sum()}")
print(f"UNPAID: {(~all_orders_overview['is_paid']).sum()}")

CREATING ALL ORDERS OVERVIEW

Total orders: 42
PAID: 29
UNPAID: 13


In [11]:
# Export to Excel - FINAL VERSION
print("="*70)
print("EXPORTING TO EXCEL")
print("="*70)

excel_filename = 'marketplace_financial_analysis.xlsx'
writer = pd.ExcelWriter(excel_filename, engine='xlsxwriter')

# Tab 1: Summary
print("\n1. Summary...")
summary_df.to_excel(writer, sheet_name='1_Summary', startrow=0)
additional_info = pd.DataFrame({
    'Metric': [
        'Actual Payout Received (Freigabe)',
        'Expected Payout (Paid)',
        'Difference',
        'Expected Payout (Unpaid/Pending)',
        'Avg Commission Rate (%)',
        'Total Commissions',
        'Orders with Missing Amounts',
        'Unpaid Orders Count',
        'Shipped Orders',
        'Cancelled/Returned (incl fees)'
    ],
    'Value': [
        abs(actual_received_freigabe),
        paid_orders['payout'].sum(),
        abs(actual_received_freigabe) - paid_orders['payout'].sum(),
        unpaid_orders['payout'].sum(),
        avg_commission_rate,
        all_orders['fee_gross'].sum(),
        len(problematic_orders),
        len(unpaid_orders),
        len(shipped_orders),
        len(cancelled_returned_orders) + len(cancelled_fees)
    ]
})
additional_info.to_excel(writer, sheet_name='1_Summary', startrow=len(summary_df)+3, index=False)

# Tab 2: ALL ORDERS OVERVIEW
print("2. All Orders Overview...")
all_orders_export = all_orders_overview[[
    'order_number', 'payment_status', 'source', 'booking_date', 'order_date',
    'title_item', 'price_gross', 'sum_price_gross', 'fee_gross',
    'payout', 'payment_received_date', 'days_since_order'
]].copy()
all_orders_export.to_excel(writer, sheet_name='2_All_Orders_Overview', index=False)

# Tab 3: Unpaid Orders
print("3. Unpaid Orders...")
if len(unpaid_orders) > 0:
    unpaid_export = unpaid_orders[[
        'booking_date', 'order_date', 'order_number', 'source', 'title_item',
        'price_gross', 'sum_price_gross', 'fee_gross', 'payout'
    ]].copy().sort_values('booking_date', ascending=False)
    unpaid_export['days_since_order'] = (pd.Timestamp.now() - unpaid_export['booking_date']).dt.days
    unpaid_export.to_excel(writer, sheet_name='3_Unpaid_Orders_Detail', index=False)
else:
    pd.DataFrame({'Message': ['No unpaid orders']}).to_excel(writer, sheet_name='3_Unpaid_Orders_Detail', index=False)

# Tab 4: Paid Orders
print("4. Paid Orders...")
paid_export = paid_orders[[
    'booking_date', 'order_date', 'order_number', 'title_item',
    'price_gross', 'sum_price_gross', 'fee_gross', 'payout'
]].copy().sort_values('booking_date', ascending=False)
paid_export.to_excel(writer, sheet_name='4_Paid_Orders_Detail', index=False)

# Tab 5: Shipped Orders
print("5. Shipped Orders...")
shipped_export = shipped_orders[[
    'booking_date', 'order_date', 'order_number', 'title_item',
    'price_gross', 'sum_price_gross', 'fee_gross', 'payout',
    'payment_status'
]].copy().sort_values('booking_date', ascending=False)
shipped_export.to_excel(writer, sheet_name='5_Shipped_Orders', index=False)

# Tab 6: Cancelled/Returned (including fees)
print("6. Cancelled/Returned...")
if len(cancelled_returned_orders) > 0 or len(cancelled_fees) > 0:
    # Combine cancelled orders and fees
    cancelled_combined = []
    
    # Add cancelled orders
    if len(cancelled_returned_orders) > 0:
        for _, row in cancelled_returned_orders.iterrows():
            cancelled_combined.append({
                'booking_date': row['booking_date'],
                'order_number': row['order_number'],
                'type': 'Cancelled Order',
                'title_item': row.get('title_item', ''),
                'amount': row['payout'],
                'booking_text': row['booking_text']
            })
    
    # Add cancelled fees
    if len(cancelled_fees) > 0:
        for _, row in cancelled_fees.iterrows():
            cancelled_combined.append({
                'booking_date': row['booking_date'],
                'order_number': 'N/A',
                'type': 'Cancelled Fee',
                'title_item': 'N/A',
                'amount': row['payout'],
                'booking_text': row['booking_text']
            })
    
    cancelled_export = pd.DataFrame(cancelled_combined).sort_values('booking_date', ascending=False)
    cancelled_export.to_excel(writer, sheet_name='6_Cancelled_Returned', index=False)
else:
    pd.DataFrame({'Message': ['No cancelled/returned']}).to_excel(writer, sheet_name='6_Cancelled_Returned', index=False)

# Tab 7: Payment Timing
print("7. Payment Timing...")
timing_export = paid_details[[
    'order_number', 'booking_date', 'order_created_date', 'payment_received_date',
    'payment_delay_days', 'price_gross', 'payout'
]].copy().sort_values('payment_received_date', ascending=False)
timing_export.to_excel(writer, sheet_name='7_Payment_Timing', index=False)

# Tab 8: Monthly GMU Categories
print("8. Monthly GMU Categories...")
gmu_cat_export = gmu_category_monthly.copy()
gmu_cat_export.index = gmu_cat_export.index.astype(str)
gmu_cat_export.to_excel(writer, sheet_name='8_Monthly_GMU_Categories')

# Tab 9: Monthly Payment Categories (CORRECTED SIGNS)
print("9. Monthly Payment Categories...")
df10k_cat_export = df10000_category_monthly.copy()
df10k_cat_export.index = df10k_cat_export.index.astype(str)
df10k_cat_export.to_excel(writer, sheet_name='9_Monthly_Payment_Categories')

# Tab 10: Missing Amounts ERROR
print("10. Missing Amounts...")
if len(problematic_orders) > 0:
    prob_export = problematic_orders[[
        'booking_date', 'order_number', 'booking_text', 'price_gross',
        'sum_price_gross', 'payout', 'has_missing_price',
        'has_missing_payout', 'has_missing_sum'
    ]].copy().sort_values('booking_date', ascending=False)
    prob_export.to_excel(writer, sheet_name='10_Missing_Amounts_ERROR', index=False)
else:
    pd.DataFrame({'Message': ['No missing amounts']}).to_excel(writer, sheet_name='10_Missing_Amounts_ERROR', index=False)

# Tab 11: All GMU Transactions
print("11. All GMU Transactions...")
gmu_export = df_gmu.copy()
gmu_export['month'] = gmu_export['month'].astype(str)
gmu_export.to_excel(writer, sheet_name='11_All_GMU_Transactions', index=False)

# Tab 12: All Payments (df_10000)
print("12. All Payments Received...")
df10k_export = df_10000.copy()
df10k_export['month'] = df10k_export['month'].astype(str)
df10k_export = df10k_export.sort_values('Datum', ascending=False)
df10k_export.to_excel(writer, sheet_name='12_All_Payments_Received', index=False)

writer.close()

print(f"\n✓ Excel file created: {excel_filename}")
print("\n✅ FIXES APPLIED:")
print("  - Unpaid orders now include orders from df_10000 (like MHFBPSQ)")
print("  - Monthly Payment Categories: signs corrected (positive for easy reading)")
print("  - Cancelled_Returned: now includes 'Fees for cancelled orders' from GMU")
print("  - All orders have 'source' column (GMU or df_10000_only)")

EXPORTING TO EXCEL

1. Summary...
2. All Orders Overview...
3. Unpaid Orders...
4. Paid Orders...
5. Shipped Orders...
6. Cancelled/Returned...
7. Payment Timing...
8. Monthly GMU Categories...
9. Monthly Payment Categories...
10. Missing Amounts...
11. All GMU Transactions...
12. All Payments Received...

✓ Excel file created: marketplace_financial_analysis.xlsx

✅ FIXES APPLIED:
  - Unpaid orders now include orders from df_10000 (like MHFBPSQ)
  - Monthly Payment Categories: signs corrected (positive for easy reading)
  - Cancelled_Returned: now includes 'Fees for cancelled orders' from GMU
  - All orders have 'source' column (GMU or df_10000_only)
